# 02 — Baseline Models

**Purpose:** Establish honest baseline F1 scores before training the Random Forest.  
The RF (notebook 03) must beat these baselines or the thesis has a problem.

**Input:** `training/splits/train.parquet`, `training/splits/val.parquet`  
**Output:** Baseline F1 table saved to `training/results/baseline_results.csv`

**Rule R2:** All evaluation in this notebook uses the **val** set only. Test set is locked.

## 1. Load Features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

SPLITS = Path("../../training/splits")
RESULTS = Path("../../training/results")
RESULTS.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(SPLITS / "train.parquet")
val_df   = pd.read_parquet(SPLITS / "val.parquet")

DROP_COLS = [
    "status_code", "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s", "_source",
    "sample_id", "timestamp", "_row_hash",
]
train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns], inplace=True)
val_df.drop(columns=[c for c in DROP_COLS if c in val_df.columns], inplace=True)

y_train = train_df.pop("label")
y_val   = val_df.pop("label")
X_train, X_val = train_df, val_df

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print("Train class distribution:")
print(y_train.value_counts())

## 2. Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

TARGET_NAMES = ["benign", "cmdi", "path_traversal", "sqli", "xss"]

print("=== Logistic Regression ===")
print(classification_report(y_val, y_pred_lr, target_names=TARGET_NAMES))
print(f"Macro F1: {f1_score(y_val, y_pred_lr, average='macro'):.4f}")

## 3. Decision Tree Baseline

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42,
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_val)

print("=== Decision Tree (max_depth=10) ===")
print(classification_report(y_val, y_pred_dt, target_names=TARGET_NAMES))
print(f"Macro F1: {f1_score(y_val, y_pred_dt, average='macro'):.4f}")

## 4. Results Table

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (max_depth=10)"],
    "Macro F1 (val)": [
        f1_score(y_val, y_pred_lr, average="macro"),
        f1_score(y_val, y_pred_dt, average="macro"),
    ],
})

out_path = RESULTS / "baseline_results.csv"
results.to_csv(out_path, index=False)

print(results.to_string(index=False))
print(f"\nSaved to {out_path}")
print("Random Forest (notebook 03) must exceed these numbers.")